# Laboratório: Pipeline de Conteúdo para LinkedIn com LangChain + Groq

## Objetivo

Construiremos um pipeline de criação de conteúdo:

1. Um modelo cria o rascunho.
2. Outro modelo revisa o texto.
3. Outras chains geram título, CTA e hashtags.
4. O resultado final é validado com Pydantic.

Fluxo:

```text
Entrada
  ↓
Criador
  ↓
Revisor
  ↓
Título / CTA / Hashtags
  ↓
Modelo estruturado
```


## 1. Instalação

In [ ]:
!pip -q install -U langchain langchain-groq pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.6 MB/s eta 0:00:00


## 2. Configuração Groq

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Digite sua GROQ_API_KEY: ")


Digite sua GROQ_API_KEY: ··········


## 3. Dois modelos

Utilizamos modelos diferentes para responsabilidades diferentes:

- Criador: velocidade e geração inicial.
- Revisor: qualidade e refinamento.


In [ ]:
from langchain_groq import ChatGroq

llm_criador = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7,
    max_tokens=600
)

llm_revisor = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3,
    max_tokens=700
)


## 4. Criando o post inicial

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

prompt_criador = ChatPromptTemplate.from_template(
    """
    Você é um especialista em criação de conteúdo para LinkedIn.

    Crie um post sobre:
    {tema}

    Público:
    {publico}

    Objetivo:
    {objetivo}

    Tom:
    {tom}

    Regras:
    - máximo de 180 palavras;
    - utilize parágrafos curtos;
    - não crie título;
    - não crie hashtags.
    """
)

chain_criador = prompt_criador | llm_criador | parser

entrada = {
    "tema": "IA Generativa transformando profissionais de dados",
    "publico": "profissionais de tecnologia",
    "objetivo": "gerar reflexão",
    "tom": "profissional e educativo"
}

rascunho = chain_criador.invoke(entrada)

print(rascunho)


A IA Generativa está mudando o jogo para profissionais de dados. Com a capacidade de criar conteúdo, imagens e até mesmo áudio a partir de dados, a IA está proporcionando novas oportunidades de trabalho e desafios para os especialistas em análise de dados.

A IA Generativa pode ser usada para criar modelos de previsão mais precisos, ajudando os profissionais de dados a tomar decisões informadas e estratégicas. Além disso, ela pode ser usada para automatizar tarefas repetitivas, liberando tempo para que os profissionais de dados se concentrem em análises mais complexas e criativas.

No entanto, a IA Generativa também está gerando preocupações sobre a substituição de empregos e a necessidade de desenvolver habilidades novas. Como profissionais de tecnologia, é importante estar preparados para se adaptar a essas mudanças e encontrar novas formas de se valorizar no mercado de trabalho. Estamos prontos para responder ao desafio?


## 5. Revisão utilizando um segundo modelo

In [ ]:
prompt_revisor = ChatPromptTemplate.from_template(
    """
    Você é um editor profissional de LinkedIn.

    Revise o texto abaixo:

    {texto}

    Faça:
    - correção gramatical;
    - melhoria de clareza;
    - melhoria de impacto;
    - mantenha a ideia original.

    Retorne somente o texto final revisado.
    """
)

chain_revisor = prompt_revisor | llm_revisor | parser

post_revisado = chain_revisor.invoke({
    "texto": rascunho
})

print("POST REVISADO")
print("=" * 60)
print(post_revisado)


POST REVISADO
IA Generativa está transformando o cenário para profissionais de dados. Ao criar conteúdo, imagens e até áudio a partir de dados, ela abre novas oportunidades e traz desafios para analistas.

Com modelos de previsão mais precisos, a IA facilita decisões informadas e estratégicas. Além disso, automatiza tarefas repetitivas, liberando tempo para análises mais complexas e criativas.

Entretanto, a tecnologia também desperta preocupações sobre a substituição de empregos e a necessidade de novas competências. Como profissionais de tecnologia, devemos nos adaptar e descobrir maneiras de agregar valor no mercado. Você está pronto para o desafio?


## 6. Gerando elementos complementares em paralelo

In [ ]:
from langchain_core.runnables import RunnableParallel

prompt_titulo = ChatPromptTemplate.from_template(
    "Crie um título curto para este post:\n{post}"
)

prompt_cta = ChatPromptTemplate.from_template(
    "Crie uma pergunta final para gerar comentários neste post:\n{post}"
)

prompt_hashtags = ChatPromptTemplate.from_template(
    "Gere cinco hashtags para este post:\n{post}"
)

chain_complementos = RunnableParallel(
    titulo=prompt_titulo | llm_criador | parser,
    cta=prompt_cta | llm_criador | parser,
    hashtags=prompt_hashtags | llm_criador | parser
)

complementos = chain_complementos.invoke({
    "post": post_revisado
})

print(complementos)


{'titulo': '"A Era da IA: Oportunidades e Desafios para Analistas de Dados"\n\nou\n\n"IA Generativa: Transformando o Cenário de Dados em 5 Linhas"', 'cta': 'Aqui vai uma pergunta final para gerar comentários:\n\n"Que habilidades ou ferramentas você acredita serem essenciais para profissionais de dados para se manterem atualizados e competitivos no mercado de trabalho com a crescente adoção da IA Generativa?"', 'hashtags': 'Aqui estão cinco hashtags que podem ser utilizados para o post:\n\n1. #IAGenerativa\n2. #TecnologiaEvoluindo\n3. #ProfissionaisDeDados\n4. #TransformaçãoDigital\n5. #InovaçãoNaAnáliseDeDados'}


## 7. Saída estruturada com Pydantic

Aqui utilizaremos Pydantic para validar o formato final.

Neste cenário, não chamaremos outro modelo apenas para organizar dados.
O próprio pipeline já produziu as informações necessárias.


In [ ]:
from pydantic import BaseModel, Field

class ConteudoLinkedIn(BaseModel):
    titulo: str = Field(description="Título")
    post: str = Field(description="Texto revisado")
    cta: str = Field(description="Chamada para ação")
    hashtags: list[str] = Field(description="Hashtags")

resultado_final = ConteudoLinkedIn(
    titulo=complementos["titulo"],
    post=post_revisado,
    cta=complementos["cta"],
    hashtags=complementos["hashtags"].replace(",", " ").split()
)

print("CONTEÚDO FINAL")
print("=" * 60)
print("Título:")
print(resultado_final.titulo)

print("\nPost:")
print(resultado_final.post)

print("\nCTA:")
print(resultado_final.cta)

print("\nHashtags:")
print(resultado_final.hashtags)


CONTEÚDO FINAL
Título:
"A Era da IA: Oportunidades e Desafios para Analistas de Dados"

ou

"IA Generativa: Transformando o Cenário de Dados em 5 Linhas"

Post:
IA Generativa está transformando o cenário para profissionais de dados. Ao criar conteúdo, imagens e até áudio a partir de dados, ela abre novas oportunidades e traz desafios para analistas.

Com modelos de previsão mais precisos, a IA facilita decisões informadas e estratégicas. Além disso, automatiza tarefas repetitivas, liberando tempo para análises mais complexas e criativas.

Entretanto, a tecnologia também desperta preocupações sobre a substituição de empregos e a necessidade de novas competências. Como profissionais de tecnologia, devemos nos adaptar e descobrir maneiras de agregar valor no mercado. Você está pronto para o desafio?

CTA:
Aqui vai uma pergunta final para gerar comentários:

"Que habilidades ou ferramentas você acredita serem essenciais para profissionais de dados para se manterem atualizados e competitivo

## Conceitos demonstrados

| Conceito | Uso |
|---|---|
| ChatGroq | Uso de múltiplos modelos |
| Prompt Template | Prompts reutilizáveis |
| LCEL | Composição com `|` |
| StrOutputParser | Texto de saída |
| RunnableParallel | Execução paralela |
| Pydantic | Validação estrutural |

## Desafio

Adapte o pipeline para gerar uma newsletter, artigo de blog ou descrição de produto.
